# Module 6: Hedge Liquidity Mapping Staging

This notebook creates the hedge-to-liquidity mapping staging tables in `main.regtech_ops_stg`.

**Replicates SSIS package:** `HedgeServerToLiquidity_Mapping.dtsx` (truncate/reload)

**Targets created:**
- `bi_output_regtechops_reg_hedgeservertoliquidityaccount_ext` — HedgeServer-to-LiquidityAccount mapping
- `bi_output_regtechops_reg_liquidtyacount_ext` — Liquidity accounts (sensitive cols excluded)
- `bi_output_regtechops_reg_ext_liquidityaccountid` — LiquidityAccountID-to-LEI mapping (Google Sheets)
- `bi_output_regtechops_reg_ext_liquidityproviders` — Liquidity providers with type names

**Note:** `Reg_LiquidtyAcount_SCD` (incremental SCD) is handled separately after initial seed decision.

In [0]:
dbutils.widgets.text("report_date", "2026-06-09", "Report Date (YYYY-MM-DD)")

report_date = dbutils.widgets.get("report_date")
print(f"Running Module 6: Hedge Liquidity Staging for report_date = {report_date}")

In [0]:
%sql
-- Reg_HedgeServerToLiquidityAccount_Ext: HedgeServer-to-LiquidityAccount mapping
-- SSIS parity: full snapshot from Hedge.HedgeServerToLiquidityAccount
-- Source: main.bi_db.bronze_etoro_hedge_hedgeservertoliquidityaccount (confirmed accessible)

CREATE OR REPLACE TABLE main.regtech_ops_stg.bi_output_regtechops_reg_hedgeservertoliquidityaccount_ext
USING DELTA
LOCATION 'abfss://analysis@stgdpdlwe.dfs.core.windows.net/BI_OUTPUT/RegTechOps/reg_hedgeservertoliquidityaccount_ext'
AS
SELECT
  HedgeServerID,
  LiquidityAccountID,
  AltRatesLiquidityAccountID
FROM main.bi_db.bronze_etoro_hedge_hedgeservertoliquidityaccount

In [0]:
%sql
-- Reg_LiquidtyAcount_Ext: Liquidity accounts (sensitive columns excluded per Phase 1 policy)
-- SSIS parity: Trade.LiquidityAccounts snapshot
-- Source: main.trading.bronze_etoro_trade_liquidityaccounts
-- Note: Password, SettingsXML, Username intentionally excluded (not required by SP_Reg_LiquidtyAcount_SCD)

CREATE OR REPLACE TABLE main.regtech_ops_stg.bi_output_regtechops_reg_liquidtyacount_ext
USING DELTA
LOCATION 'abfss://analysis@stgdpdlwe.dfs.core.windows.net/BI_OUTPUT/RegTechOps/reg_liquidtyacount_ext'
AS
SELECT
  LiquidityProviderID,
  LiquidityAccountID,
  LiquidityAccountName,
  CAST(IsActive AS INT) AS IsActive,
  LiquidityAccountTypeID,
  AccountRateSourceID
FROM main.trading.bronze_etoro_trade_liquidityaccounts

In [0]:
%sql
-- Reg_Ext_LiquidityAccountID: LiquidityAccountID-to-LEI mapping from Google Sheets
-- Source: main.general.bronze_fivetran_google_sheets_reg_liquidityaccountid_to_lei
-- SSIS parity: manual reference table; Fivetran-ingested equivalent

CREATE OR REPLACE TABLE main.regtech_ops_stg.bi_output_regtechops_reg_ext_liquidityaccountid
USING DELTA
LOCATION 'abfss://analysis@stgdpdlwe.dfs.core.windows.net/BI_OUTPUT/RegTechOps/reg_ext_liquidityaccountid'
AS
SELECT
  liquidity_account_id AS LiquidityAccountID,
  liquidity_account_name AS LiquidityAccountName,
  is_active AS IsActive,
  e_toro_entity AS eToroEntity,
  real_or_cfd AS RealOrCFD,
  lei AS LEI,
  lp_country_code AS LpCountryCode,
  trading_account_purpose_or_traded_instruments AS Comment,
  to_utc_timestamp(current_timestamp(), current_timezone()) AS UpdateDate
FROM main.general.bronze_fivetran_google_sheets_reg_liquidityaccountid_to_lei

In [0]:
%sql
-- Reg_Ext_LiquidityProviders: Liquidity providers with type names
-- Source: main.trading.bronze_etoro_trade_liquidityproviders + main.bi_db.bronze_etoro_trade_liquidityprovidertype
-- SSIS parity: joined provider + provider type snapshot

CREATE OR REPLACE TABLE main.regtech_ops_stg.bi_output_regtechops_reg_ext_liquidityproviders
USING DELTA
LOCATION 'abfss://analysis@stgdpdlwe.dfs.core.windows.net/BI_OUTPUT/RegTechOps/reg_ext_liquidityproviders'
AS
SELECT
  lp.LiquidityProviderID,
  lp.LiquidityProviderName,
  lp.LiquidityProviderTypeID,
  lpt.Name AS LiquidityProviderTypeName,
  to_utc_timestamp(current_timestamp(), current_timezone()) AS UpdateDate
FROM main.trading.bronze_etoro_trade_liquidityproviders lp
JOIN main.bi_db.bronze_etoro_trade_liquidityprovidertype lpt
  ON lp.LiquidityProviderTypeID = lpt.LiquidityProviderTypeID

In [0]:
%sql
-- Validation: row counts for all Module 6 staging tables
SELECT 'reg_hedgeservertoliquidityaccount_ext' AS table_name, COUNT(*) AS row_count FROM main.regtech_ops_stg.bi_output_regtechops_reg_hedgeservertoliquidityaccount_ext
UNION ALL SELECT 'reg_liquidtyacount_ext', COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_liquidtyacount_ext
UNION ALL SELECT 'reg_ext_liquidityaccountid', COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_ext_liquidityaccountid
UNION ALL SELECT 'reg_ext_liquidityproviders', COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_ext_liquidityproviders

# Module 6: Hedge Liquidity Mapping Staging

This notebook creates the hedge/liquidity mapping staging tables in `main.regtech_ops_stg`.

**Replicates SSIS package:** `HedgeServerToLiquidity_Mapping.dtsx` (truncate/reload)

**Targets created:**
- `bi_output_regtechops_reg_hedgeservertoliquidityaccount_ext` — HedgeServer to LiquidityAccount mapping
- `bi_output_regtechops_reg_liquidtyacount_ext` — LiquidityAccounts reference (sensitive fields excluded)
- `bi_output_regtechops_reg_ext_liquidityaccountid` — LiquidityAccountID to LEI mapping (Google Sheets)
- `bi_output_regtechops_reg_ext_liquidityproviders` — LiquidityProviders reference

**Note:** `Reg_LiquidtyAcount_SCD` is an incremental SCD table and requires a separate seed/cutover decision.

In [0]:
dbutils.widgets.text("report_date", "2026-06-09", "Report Date (YYYY-MM-DD)")

report_date = dbutils.widgets.get("report_date")
print(f"Running Module 6: Hedge Liquidity Staging for report_date = {report_date}")

In [0]:
%sql
-- Reg_HedgeServerToLiquidityAccount_Ext: HedgeServer to LiquidityAccount mapping
-- SSIS parity: full snapshot from Hedge.HedgeServerToLiquidityAccount
-- Source: main.bi_db.bronze_etoro_hedge_hedgeservertoliquidityaccount

CREATE OR REPLACE TABLE main.regtech_ops_stg.bi_output_regtechops_reg_hedgeservertoliquidityaccount_ext
USING DELTA
LOCATION 'abfss://analysis@stgdpdlwe.dfs.core.windows.net/BI_OUTPUT/RegTechOps/reg_hedgeservertoliquidityaccount_ext'
AS
SELECT
  HedgeServerID,
  LiquidityAccountID,
  AltRatesLiquidityAccountID
FROM main.bi_db.bronze_etoro_hedge_hedgeservertoliquidityaccount